# Solo run on Kaggle (2 x T4 GPUs, runs in the background)

Settings (right-hand panel): **Accelerator = GPU T4 x2**, **Internet = On**.
Edit `REPO_URL` in cell 1, then use **Save Version -> Save & Run All (Commit)**. You can close the browser; it keeps running (up to 12 h).
When it finishes, open the version -> **Output** tab -> download `results_all.zip`.

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/lskd.git"   # <-- change this
PHASE = 1        # 1 = reproduction + ablation (must do). 2 = CIFAR-10 + temperature (optional, second session)
EPOCHS = 240     # use 120 for EVERY run only if time is short (say so in the report)
import os
os.chdir("/kaggle/working")
!rm -rf lskd && git clone -q {REPO_URL} lskd
os.chdir("/kaggle/working/lskd")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python scripts/download_teacher.py
!python tests/test_core.py
!python train.py --preset c100_kd_ls --smoke

In [ ]:
# Two training queues, one per GPU, running at the same time. Logs: gpu0.log / gpu1.log
if PHASE == 1:
    q0 = ["c100_kd", "abl_raw_w0.9", "abl_z_w0.9"]
    q1 = ["c100_kd_ls", "abl_raw_w9", "c100_student_ce"]
else:
    q0 = ["c10_teacher_resnet56", "c10_kd", "tau1_kd_ls"]
    q1 = ["tau4_kd_ls"]          # c10_kd_ls needs the c10 teacher -> runs after it, see next cell
def chain(gpu, presets):
    return " && ".join(f"CUDA_VISIBLE_DEVICES={gpu} python train.py --preset {p} --epochs {EPOCHS} --resume" for p in presets)
script = f"({chain(0, q0)}) > gpu0.log 2>&1 &\n({chain(1, q1)}) > gpu1.log 2>&1 &\nwait\n"
open("queue.sh", "w").write(script)
print(script)
!bash queue.sh
!tail -n 3 gpu0.log gpu1.log

In [ ]:
if PHASE == 2:
    !python train.py --preset c10_kd_ls --epochs {EPOCHS} --resume > gpu_c10ls.log 2>&1
    !tail -n 3 gpu_c10ls.log

In [ ]:
# Evaluation + report tables/figures
if os.path.exists("runs/c100_kd/final.pth") and os.path.exists("runs/c100_kd_ls/final.pth"):
    !python evaluate.py --dataset cifar100 --teacher checkpoints/teachers/resnet56_cifar100.pth --models "Vanilla KD=runs/c100_kd/final.pth" "KD+LS=runs/c100_kd_ls/final.pth"
if os.path.exists("runs/c10_kd/final.pth") and os.path.exists("runs/c10_kd_ls/final.pth"):
    !python evaluate.py --dataset cifar10 --teacher runs/c10_teacher_resnet56/final.pth --models "Vanilla KD=runs/c10_kd/final.pth" "KD+LS=runs/c10_kd_ls/final.pth"
!python scripts/make_report_assets.py
!cd /kaggle/working/lskd && zip -q -r /kaggle/working/results_all.zip runs results report/generated report/figures gpu*.log -x "*/last.pth" "*_smoke/*"
!ls -lh /kaggle/working/results_all.zip